# 🏦 SGCI — Framework Data Quality + Journalisation
### SchemaValidator · 6 Dimensions DQ · Log structuré · Teradata
---
**Auteur** : Data Engineering SGCI &nbsp;|&nbsp; **Version** : 3.0 — Juin 2026

---
## 📋 Table des matières

| # | Section | Description |
|---|---|---|
| 0 | [Configuration](#0) | Imports, logger, constantes |
| 1 | [Connexion Teradata](#1) | Connexion sécurisée + extraction |
| 2 | [SchemaValidator](#2) | Validation structurelle (8 contrôles) |
| 3 | [6 Dimensions DQ](#3) | Complétude · Unicité · Validité · Exactitude · Cohérence · Fraîcheur |
| 4 | [Pipeline principal](#4) | Orchestration complète en un appel |
| 5 | [Journalisation](#5) | Log CSV structuré + rapport Excel |


## 0. Configuration <a id='0'></a>

In [ ]:
# ═══════════════════════════════════════════════════════════
#  IMPORTS
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import logging
import os
import re
import time
import warnings
from datetime import datetime, date, timedelta
from pathlib import Path
import pytz

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════
#  CONSTANTES GLOBALES — À adapter selon ton environnement
# ═══════════════════════════════════════════════════════════

# Répertoire de logs (créé automatiquement s'il n'existe pas)
LOG_DIR       = "./JOURNAL/"

# Nom du fichier journal CSV
LOG_FILE_NAME = "journal_dq.csv"

# Nom du fichier de log texte
LOG_PROJECT   = "sgci_dq_framework"

# Fuseau horaire Abidjan (UTC+0, pas d'heure d'été)
TZ_ABIDJAN    = pytz.timezone("Africa/Abidjan")

# Détection automatique de l'utilisateur système
current_directory = os.getcwd()
USER = current_directory.split("/")[2] if len(current_directory.split("/")) > 2 else "sgci_user"

# Référentiels métier SGCI
DEVISES_OK    = {"XOF", "EUR", "USD", "GBP", "CHF", "JPY", "CNY"}
TYPES_TRN_OK  = {"VIR", "PRE", "CHQ", "CBK", "FEE", "INT", "DIV", "SWIFT"}
TYPES_CPT_OK  = {"CC", "EP", "CS", "PEL", "CEL", "PRO"}
PATTERN_IBAN  = re.compile(r'^[A-Z]{2}\d{2}[A-Z0-9]{4,30}$')
PATTERN_REF   = re.compile(r'^[A-Z0-9]{8,35}$')

# Création du répertoire de logs
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

print("✅ Configuration chargée")
print(f"   Utilisateur : {USER}")
print(f"   Log dir     : {LOG_DIR}")
print(f"   Timezone    : {TZ_ABIDJAN}")


## 1. Connexion Teradata <a id='1'></a>

In [ ]:
# ═══════════════════════════════════════════════════════════
#  LOGGER PIPELINE
# ═══════════════════════════════════════════════════════════
def setup_pipeline_logger(project_name: str, log_dir: str) -> logging.Logger:
    """
    Configure un logger structuré avec deux sorties :
    - Fichier .log  → audit complet (tous niveaux DEBUG+)
    - Console       → affichage Jupyter (INFO+)

    Format : DATE | NIVEAU | NOM_PROJET | MESSAGE
    """
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    log_file = f"{log_dir}/{project_name}.log"

    logger = logging.getLogger(project_name)
    logger.setLevel(logging.DEBUG)

    # Éviter la duplication si cellule ré-exécutée
    if logger.handlers:
        logger.handlers.clear()

    fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    # Handler fichier
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Handler console
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    logger.propagate = False
    logger.info(f"Logger initialisé → {log_file}")
    return logger


logger = setup_pipeline_logger(LOG_PROJECT, LOG_DIR)


In [ ]:
# ═══════════════════════════════════════════════════════════
#  CONNEXION TERADATA
# ═══════════════════════════════════════════════════════════
def get_connection():
    """
    Retourne une connexion Teradata sécurisée.
    Les credentials sont lus depuis les variables d'environnement.

    Variables à définir avant d'utiliser :
        TD_HOST     → Adresse du serveur Teradata
        TD_USER     → Nom d'utilisateur
        TD_PASSWORD → Mot de passe

    Exemple de définition (à faire UNE FOIS en début de session) :
        import os
        os.environ["TD_HOST"]     = "mon_serveur_td"
        os.environ["TD_USER"]     = "mon_user"
        os.environ["TD_PASSWORD"] = "mon_mdp"
    """
    import teradatasql  # pip install teradatasql
    return teradatasql.connect(
        host     = os.getenv("TD_HOST"),
        user     = os.getenv("TD_USER"),
        password = os.getenv("TD_PASSWORD"),
        logmech  = "LDAP"   # Remplacer par "TD2" si authentification native
    )


def extraire_depuis_teradata(query: str, nom: str) -> pd.DataFrame:
    """
    Exécute une requête SQL sur Teradata et retourne un DataFrame.

    Args:
        query : Requête SQL complète
        nom   : Nom descriptif pour les logs

    Returns:
        pd.DataFrame avec les résultats
    """
    logger.info(f"Extraction '{nom}' — démarrage")
    t0 = time.time()

    with get_connection() as con:
        df = pd.read_sql(query, con)

    duree = round(time.time() - t0, 2)
    logger.info(f"Extraction '{nom}' — {len(df):,} lignes × {df.shape[1]} colonnes en {duree}s")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# REQUÊTES SQL — À personnaliser selon tes tables
# ─────────────────────────────────────────────────────────────────────────────
QUERY_TRANSACTIONS = """
    SELECT
        t.TRN_REF,
        t.ACCOUNT_ID,
        t.AMOUNT,
        t.CURRENCY,
        t.TRANSACTION_TYPE,
        t.TRANSACTION_SIDE,
        t.VALUE_DATE,
        t.BOOKING_DATE,
        t.COUNTERPART_IBAN,
        t.BALANCE,
        a.ACCOUNT_TYPE
    FROM SGCI_DB.TRANSACTIONS t
    LEFT JOIN SGCI_DB.ACCOUNTS a
           ON t.ACCOUNT_ID = a.ACCOUNT_ID
    WHERE t.BOOKING_DATE = CURRENT_DATE - 1
"""

QUERY_COMPTES = """
    SELECT ACCOUNT_ID, ACCOUNT_TYPE, STATUS
    FROM   SGCI_DB.ACCOUNTS
    WHERE  STATUS = 'ACTIVE'
"""

# En production : décommenter les deux lignes ci-dessous
# df_raw     = extraire_depuis_teradata(QUERY_TRANSACTIONS, "transactions_J1")
# df_comptes = extraire_depuis_teradata(QUERY_COMPTES, "referentiel_comptes")

print("✅ Fonctions de connexion définies")
print("   → Décommenter les appels ci-dessus pour utiliser en production")


In [ ]:
# ═══════════════════════════════════════════════════════════
#  MODE DÉMONSTRATION — Données simulées
#  Remplacer par extraire_depuis_teradata() en production
# ═══════════════════════════════════════════════════════════
def simuler_extraction(n: int = 300) -> tuple:
    """
    Génère (df_transactions, df_comptes) simulés avec anomalies injectées.
    Reproduit fidèlement la structure des tables SGCI réelles.
    """
    np.random.seed(2026)
    hier = datetime.now().date() - timedelta(days=1)

    df_trn = pd.DataFrame({
        "TRN_REF"          : [f"TRN{str(i).zfill(8)}" for i in range(n)],
        "ACCOUNT_ID"       : [f"SGC{np.random.randint(10000, 99999)}" for _ in range(n)],
        "AMOUNT"           : np.random.lognormal(11, 2, n).round(2),
        "CURRENCY"         : np.random.choice(["XOF"]*88 + ["EUR"]*7 + ["USD"]*5, n),
        "TRANSACTION_TYPE" : np.random.choice(["VIR","PRE","CHQ","CBK","FEE"], n),
        "TRANSACTION_SIDE" : np.random.choice(["D","C"], n),
        "VALUE_DATE"       : [hier] * n,
        "BOOKING_DATE"     : [hier] * n,
        "COUNTERPART_IBAN" : [f"CI{np.random.randint(10,99)}{np.random.randint(10**10,10**11)}" for _ in range(n)],
        "BALANCE"          : np.random.lognormal(13, 2, n).round(2),
        "ACCOUNT_TYPE"     : np.random.choice(["CC","EP","CS","PRO"], n),
    })

    # Anomalies injectées pour tester les contrôles DQ
    df_trn.loc[[5, 42, 87], "AMOUNT"]             = np.nan          # nulls critiques
    df_trn.loc[[12, 33],    "TRN_REF"]            = np.nan          # clé primaire nulle
    df_trn = pd.concat([df_trn, df_trn.iloc[[0,1,2]]], ignore_index=True)  # doublons
    df_trn.loc[df_trn.index[-4:-2], "CURRENCY"]   = "CFA"           # devise invalide
    df_trn.loc[df_trn.index[-2],    "AMOUNT"]      = -75000.0       # montant négatif
    df_trn.loc[[20, 21], "VALUE_DATE"] = hier - timedelta(days=90)  # date incohérente

    df_cpt = pd.DataFrame({
        "ACCOUNT_ID"   : df_trn["ACCOUNT_ID"].unique()[:250],
        "ACCOUNT_TYPE" : np.random.choice(["CC","EP","CS","PRO"], 250),
        "STATUS"       : "ACTIVE"
    })

    return df_trn, df_cpt


df_raw, df_comptes = simuler_extraction(n=300)

logger.info(f"Données chargées : {len(df_raw):,} transactions | {len(df_comptes):,} comptes")
print(f"\n📊 Données chargées :")
print(f"   Transactions : {len(df_raw):,} lignes × {df_raw.shape[1]} colonnes")
print(f"   Comptes ref  : {len(df_comptes):,} lignes × {df_comptes.shape[1]} colonnes")
df_raw.head(4)


## 2. SchemaValidator <a id='2'></a>

> Validation **structurelle** du DataFrame avant tout traitement DQ.  
> API fluente (chaînable) — chaque méthode retourne `self`.

In [ ]:
class SchemaValidator:
    """
    Validateur de schéma pour DataFrames bancaires.

    Erreurs   → bloquantes  → status = FAIL  → pipeline arrêté
    Warnings  → informatifs → status = PASS  → pipeline continue avec alertes

    Usage :
        sv = (SchemaValidator("ma_table", logger)
              .check_required_columns(df, ["col1", "col2"])
              .check_no_null(df, ["col1"])
              .check_range(df, "montant", 0, 500_000_000)
              .check_positive(df, "montant")
              .check_negative(df, "solde_debiteur")
              .check_type(df, "ref", str)
              .check_unique_by(df, ["ref_transaction"])
              .check_no_full_duplicates(df)
        )
        # NE PAS appeler .validate() ici — le pipeline le fait automatiquement
    """

    def __init__(self, name: str, logger: logging.Logger):
        self.name     = name
        self.logger   = logger
        self.errors   = []
        self.warnings = []

    # ── 1. Colonnes obligatoires ───────────────────────────────────────────────
    def check_required_columns(self, df: pd.DataFrame, cols: list) -> "SchemaValidator":
        """Vérifie que toutes les colonnes requises sont présentes dans le DataFrame."""
        missing = set(cols) - set(df.columns)
        if missing:
            self.errors.append(f"Colonnes manquantes : {sorted(missing)}")
        return self

    # ── 2. Absence de NULL ─────────────────────────────────────────────────────
    def check_no_null(self, df: pd.DataFrame, cols: list) -> "SchemaValidator":
        """Vérifie qu'aucune valeur NULL n'est présente dans les colonnes spécifiées."""
        for col in cols:
            if col in df.columns:
                n = df[col].isna().sum()
                if n > 0:
                    self.warnings.append(f"'{col}' : {n} valeur(s) NULL")
        return self

    # ── 3. Plage de valeurs ────────────────────────────────────────────────────
    def check_range(self, df: pd.DataFrame, col: str, min_val, max_val) -> "SchemaValidator":
        """Vérifie que les valeurs sont dans l'intervalle [min_val, max_val]."""
        if col not in df.columns:
            return self
        out = df[df[col].notna() & ((df[col] < min_val) | (df[col] > max_val))]
        if len(out):
            self.warnings.append(f"'{col}' : {len(out)} valeur(s) hors plage [{min_val}, {max_val}]")
        return self

    # ── 4. Positivité ──────────────────────────────────────────────────────────
    def check_positive(self, df: pd.DataFrame, col: str) -> "SchemaValidator":
        """Vérifie que toutes les valeurs non nulles sont >= 0."""
        if col not in df.columns:
            return self
        neg = df[df[col].notna() & (df[col] < 0)]
        if len(neg):
            self.warnings.append(f"'{col}' : {len(neg)} valeur(s) négative(s)")
        return self

    # ── 5. Négativité ──────────────────────────────────────────────────────────
    def check_negative(self, df: pd.DataFrame, col: str) -> "SchemaValidator":
        """Vérifie que toutes les valeurs non nulles sont <= 0."""
        if col not in df.columns:
            return self
        pos = df[df[col].notna() & (df[col] > 0)]
        if len(pos):
            self.warnings.append(f"'{col}' : {len(pos)} valeur(s) positive(s)")
        return self

    # ── 6. Type de données ─────────────────────────────────────────────────────
    def check_type(self, df: pd.DataFrame, col: str, expected_type) -> "SchemaValidator":
        """Vérifie que les valeurs correspondent au type Python attendu."""
        if col not in df.columns:
            return self
        wrong = df[df[col].notna() & ~df[col].apply(lambda v: isinstance(v, expected_type))]
        if len(wrong):
            self.errors.append(
                f"'{col}' : {len(wrong)} valeur(s) type incorrect (attendu: {expected_type.__name__})"
            )
        return self

    # ── 7. Unicité sur un ensemble de colonnes ─────────────────────────────────
    def check_unique_by(self, df: pd.DataFrame, cols: list) -> "SchemaValidator":
        """Vérifie l'absence de doublons sur un ensemble de colonnes (clé composite ou simple)."""
        missing = [c for c in cols if c not in df.columns]
        if missing:
            self.warnings.append(f"check_unique_by : colonnes absentes {missing}")
            return self
        dup = df[df.duplicated(subset=cols, keep=False)]
        if len(dup):
            self.warnings.append(f"Doublons sur {cols} : {len(dup)} ligne(s)")
        return self

    # ── 8. Lignes entièrement dupliquées ───────────────────────────────────────
    def check_no_full_duplicates(self, df: pd.DataFrame) -> "SchemaValidator":
        """Vérifie qu'aucune ligne complète (toutes colonnes) n'est identique à une autre."""
        dup = df[df.duplicated(keep=False)]
        if len(dup):
            self.warnings.append(f"{len(dup)} ligne(s) entièrement dupliquée(s)")
        return self

    # ── Résultat ───────────────────────────────────────────────────────────────
    def validate(self) -> dict:
        """Consolide les résultats et retourne le rapport."""
        status = "PASS" if not self.errors else "FAIL"
        result = {
            "dataset"  : self.name,
            "status"   : status,
            "errors"   : self.errors,
            "warnings" : self.warnings,
            "timestamp": datetime.now().isoformat()
        }
        if self.errors:
            self.logger.error(f"SchemaValidator '{self.name}' → FAIL | {self.errors}")
        if self.warnings:
            self.logger.warning(f"SchemaValidator '{self.name}' → {status} | {self.warnings}")
        if not self.errors and not self.warnings:
            self.logger.info(f"SchemaValidator '{self.name}' → PASS ✅")
        return result

print("✅ SchemaValidator défini (8 méthodes)")


## 3. Les 6 Dimensions Data Quality <a id='3'></a>

In [ ]:
# ═══════════════════════════════════════════════════════════
#  DIMENSION 1 — COMPLÉTUDE
# ═══════════════════════════════════════════════════════════
def verifier_completude(df: pd.DataFrame, colonnes_critiques: list,
                         seuil: float = 1.0) -> dict:
    res = {"dimension": "Completude", "statut": "OK", "details": [], "lignes_ko": pd.DataFrame()}
    ko  = set()

    for col in df.columns:
        nb_null = df[col].isnull().sum()
        nb_vide = (df[col].astype(str).str.strip() == "").sum() if df[col].dtype == object else 0
        nb_pb   = nb_null + nb_vide
        taux    = (len(df) - nb_pb) / len(df)
        statut  = "OK"

        if col in colonnes_critiques and taux < seuil:
            statut = "CRITIQUE"
            res["statut"] = "ECHEC"
            ko.update(df[df[col].isnull() | (df[col].astype(str).str.strip() == "")].index.tolist())
        elif nb_pb > 0:
            statut = "WARNING"

        res["details"].append({
            "colonne": col, "nb_nulls": int(nb_null),
            "nb_vides": int(nb_vide), "taux_remplissage": f"{taux:.1%}", "statut": statut
        })

    if ko:
        res["lignes_ko"] = df.loc[list(ko)].copy()
    return res


# ═══════════════════════════════════════════════════════════
#  DIMENSION 2 — UNICITÉ
# ═══════════════════════════════════════════════════════════
def verifier_unicite(df: pd.DataFrame, cles_primaires: list,
                      cles_secondaires: list = None) -> dict:
    res  = {"dimension": "Unicite", "statut": "OK", "doublons": pd.DataFrame(), "details": []}
    mask = df.duplicated(subset=cles_primaires, keep=False)
    nb   = mask.sum()

    if nb > 0:
        dup = df[mask].sort_values(cles_primaires)
        res.update({"statut": "ECHEC", "doublons": dup})
        res["details"].append({
            "cles": cles_primaires,
            "nb_lignes_dupliquees": int(nb),
            "nb_groupes": int(dup.groupby(cles_primaires).ngroups)
        })

    if cles_secondaires:
        for col in cles_secondaires:
            if col in df.columns:
                n2 = df[col].duplicated(keep=False).sum()
                if n2:
                    res["details"].append({"cles": [col], "nb_lignes_dupliquees": int(n2), "niveau": "WARNING"})

    return res


def deduplication_securisee(df: pd.DataFrame, cles: list,
                              strategie: str = "garder_premier") -> pd.DataFrame:
    """Déduplique en loggant les suppressions. Stratégies : garder_premier | garder_dernier | garder_tous."""
    nb_avant = len(df)
    keep_map = {"garder_premier": "first", "garder_dernier": "last"}

    if strategie in keep_map:
        df_c = df.drop_duplicates(subset=cles, keep=keep_map[strategie])
    elif strategie == "garder_tous":
        return df[df.duplicated(subset=cles, keep=False)]
    else:
        raise ValueError(f"Stratégie inconnue : {strategie}")

    logger.info(f"Déduplication sur {cles} : {nb_avant - len(df_c)} supprimé(s) ({nb_avant:,}→{len(df_c):,})")
    return df_c


# ═══════════════════════════════════════════════════════════
#  DIMENSION 3 — VALIDITÉ
#  CORRECTION : utiliser .astype(str) sur les non-nuls
#  uniquement pour éviter l'erreur .str sur types mixtes.
# ═══════════════════════════════════════════════════════════
def verifier_validite(df: pd.DataFrame) -> dict:
    res     = {"dimension": "Validite", "statut": "OK", "erreurs": [], "nb_invalides": 0}
    masques = pd.Series(False, index=df.index)
    df      = df.copy()

    # Devises — convertir en str sur les non-nuls uniquement
    if "CURRENCY" in df.columns:
        mask_nn   = df["CURRENCY"].notna()
        cur_clean = df["CURRENCY"].astype(str).str.strip().str.upper()
        m = mask_nn & ~cur_clean.isin(DEVISES_OK)
        if m.any():
            res["erreurs"].append({"champ": "CURRENCY", "nb": int(m.sum()),
                                    "valeurs": df.loc[m, "CURRENCY"].unique().tolist()})
            res["statut"] = "ECHEC"; masques |= m

    # Types de transaction
    if "TRANSACTION_TYPE" in df.columns:
        m = df["TRANSACTION_TYPE"].notna() & ~df["TRANSACTION_TYPE"].isin(TYPES_TRN_OK)
        if m.any():
            res["erreurs"].append({"champ": "TRANSACTION_TYPE", "nb": int(m.sum()),
                                    "valeurs": df.loc[m, "TRANSACTION_TYPE"].unique().tolist()})
            res["statut"] = "ECHEC"; masques |= m

    # Formats dates
    for col in ["VALUE_DATE", "BOOKING_DATE", "CREATION_DATE"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            nb_inv  = df[col].isnull().sum()
            if nb_inv > 0:
                res["erreurs"].append({"champ": col, "nb": int(nb_inv), "raison": "date invalide"})
                res["statut"] = "ECHEC"

    # IBAN — appliquer .str uniquement sur les lignes non-nulles
    if "COUNTERPART_IBAN" in df.columns:
        mask_nn = df["COUNTERPART_IBAN"].notna()
        if mask_nn.any():
            iban_str = df.loc[mask_nn, "COUNTERPART_IBAN"].astype(str).str.strip()
            inv      = ~iban_str.str.match(PATTERN_IBAN)
            m        = pd.Series(False, index=df.index)
            m[mask_nn] = inv.values
            if m.any():
                res["erreurs"].append({"champ": "COUNTERPART_IBAN", "nb": int(m.sum()),
                                        "raison": "format IBAN invalide"})

    # Référence transaction
    if "TRN_REF" in df.columns:
        mask_nn = df["TRN_REF"].notna()
        if mask_nn.any():
            ref_str = df.loc[mask_nn, "TRN_REF"].astype(str).str.strip()
            inv     = ~ref_str.str.match(PATTERN_REF)
            m       = pd.Series(False, index=df.index)
            m[mask_nn] = inv.values
            if m.any():
                res["erreurs"].append({"champ": "TRN_REF", "nb": int(m.sum()),
                                        "raison": "format REF invalide"})
                res["statut"] = "ECHEC"; masques |= m

    res["nb_invalides"] = int(masques.sum())
    return res


# ═══════════════════════════════════════════════════════════
#  DIMENSION 4 — EXACTITUDE
# ═══════════════════════════════════════════════════════════
def verifier_exactitude(df: pd.DataFrame) -> dict:
    res     = {"dimension": "Exactitude", "statut": "OK", "alertes": [], "nb_aberrants": 0}
    masques = pd.Series(False, index=df.index)
    df      = df.copy()

    if "AMOUNT" in df.columns:
        df["AMOUNT"] = pd.to_numeric(df["AMOUNT"], errors="coerce")

        mn = df["AMOUNT"] < 0
        if mn.any():
            res["alertes"].append({"type": "montant_negatif", "nb": int(mn.sum()),
                                    "min": float(df.loc[mn, "AMOUNT"].min())})
            res["statut"] = "ECHEC"; masques |= mn

        mz = df["AMOUNT"] == 0
        if mz.any():
            res["alertes"].append({"type": "montant_zero", "nb": int(mz.sum())})
            if res["statut"] == "OK": res["statut"] = "AVERTISSEMENT"

        Q1, Q3 = df["AMOUNT"].quantile(0.25), df["AMOUNT"].quantile(0.75)
        bs = Q3 + 3 * (Q3 - Q1)
        mo = df["AMOUNT"].notna() & (df["AMOUNT"] > bs)
        if mo.any():
            top5 = df.loc[mo, "AMOUNT"].nlargest(5).round(2).tolist()
            res["alertes"].append({"type": "outlier_IQR", "nb": int(mo.sum()),
                                    "borne_sup": round(bs, 2), "top5": top5})
            masques |= mo

    if "BALANCE" in df.columns:
        df["BALANCE"] = pd.to_numeric(df["BALANCE"], errors="coerce")
        mb = df["BALANCE"].notna() & (df["BALANCE"].abs() > 5_000_000_000)
        if mb.any():
            res["alertes"].append({"type": "solde_anormal", "nb": int(mb.sum())})
            res["statut"] = "ECHEC"; masques |= mb

    res["nb_aberrants"] = int(masques.sum())
    return res


# ═══════════════════════════════════════════════════════════
#  DIMENSION 5 — COHÉRENCE
#  CORRECTION : comparer les dates uniquement sur les lignes
#  où les deux colonnes sont non-nulles.
# ═══════════════════════════════════════════════════════════
def verifier_coherence(df: pd.DataFrame,
                        df_reference: pd.DataFrame = None,
                        cle_jointure: str = None) -> dict:
    res     = {"dimension": "Coherence", "statut": "OK", "anomalies": [], "nb_incoherences": 0}
    masques = pd.Series(False, index=df.index)
    df      = df.copy()

    if "VALUE_DATE" in df.columns and "BOOKING_DATE" in df.columns:
        df["VALUE_DATE"]   = pd.to_datetime(df["VALUE_DATE"],   errors="coerce")
        df["BOOKING_DATE"] = pd.to_datetime(df["BOOKING_DATE"], errors="coerce")

        # Appliquer les comparaisons uniquement où les deux dates sont valides
        mask_valid = df["VALUE_DATE"].notna() & df["BOOKING_DATE"].notna()

        m30 = mask_valid & (df["VALUE_DATE"] < (df["BOOKING_DATE"] - pd.Timedelta(days=30)))
        if m30.any():
            res["anomalies"].append({"type": "value_date_trop_ancienne", "nb": int(m30.sum())})
            res["statut"] = "ECHEC"; masques |= m30

        mfut = mask_valid & (df["VALUE_DATE"] > (pd.Timestamp.today() + pd.Timedelta(days=365)))
        if mfut.any():
            res["anomalies"].append({"type": "date_future_aberrante", "nb": int(mfut.sum())})
            res["statut"] = "ECHEC"; masques |= mfut

    if "AMOUNT" in df.columns and "TRANSACTION_SIDE" in df.columns:
        df["AMOUNT"] = pd.to_numeric(df["AMOUNT"], errors="coerce")
        totD = df.loc[df["TRANSACTION_SIDE"] == "D", "AMOUNT"].sum()
        totC = df.loc[df["TRANSACTION_SIDE"] == "C", "AMOUNT"].sum()
        if abs(totD - totC) > 1.0:
            res["anomalies"].append({"type": "desequilibre_DC",
                                      "ecart": round(abs(totD - totC), 2),
                                      "total_D": round(totD, 2), "total_C": round(totC, 2)})
            res["statut"] = "ECHEC"

    if df_reference is not None and cle_jointure and cle_jointure in df.columns:
        orph = set(df[cle_jointure].dropna()) - set(df_reference[cle_jointure].dropna())
        if orph:
            res["anomalies"].append({"type": "cle_orpheline", "champ": cle_jointure,
                                      "nb": len(orph), "exemples": list(orph)[:5]})
            res["statut"] = "ECHEC"

    res["nb_incoherences"] = int(masques.sum())
    return res


# ═══════════════════════════════════════════════════════════
#  DIMENSION 6 — FRAÎCHEUR
# ═══════════════════════════════════════════════════════════
def verifier_fraicheur(df: pd.DataFrame, col_date_ref: str,
                        date_attendue=None, tolerance_jours: int = 1) -> dict:
    res = {"dimension": "Fraicheur", "statut": "OK", "details": {}, "alertes": []}

    if date_attendue is None:
        date_attendue = (datetime.now(TZ_ABIDJAN) - timedelta(days=1)).date()

    if col_date_ref not in df.columns:
        res["statut"] = "AVERTISSEMENT"
        res["alertes"].append({"type": "colonne_absente", "champ": col_date_ref})
        return res

    df[col_date_ref] = pd.to_datetime(df[col_date_ref], errors="coerce")
    dmax = df[col_date_ref].max()
    dmin = df[col_date_ref].min()

    res["details"] = {
        "date_attendue"    : str(date_attendue),
        "date_max_extraite": str(dmax.date() if pd.notna(dmax) else "N/A"),
        "date_min_extraite": str(dmin.date() if pd.notna(dmin) else "N/A"),
        "nb_lignes"        : len(df)
    }

    if len(df) == 0:
        res["statut"] = "ECHEC"
        res["alertes"].append({"type": "extraction_vide"})
    elif pd.notna(dmax):
        ecart = (date_attendue - dmax.date()).days
        if ecart > tolerance_jours:
            res["statut"] = "ECHEC"
            res["alertes"].append({"type": "donnees_perimees",
                                    "retard_jours": ecart, "date_max": str(dmax.date())})
        elif ecart == 1:
            res["statut"] = "AVERTISSEMENT"
            res["alertes"].append({"type": "donnees_j2", "date_max": str(dmax.date())})
    else:
        res["statut"] = "ECHEC"
        res["alertes"].append({"type": "dates_nulles"})

    return res


print("✅ Les 6 fonctions DQ définies et corrigées :")
print("   verifier_completude()  | verifier_unicite()   | verifier_validite()")
print("   verifier_exactitude()  | verifier_coherence() | verifier_fraicheur()")


## 4. Pipeline principal <a id='4'></a>

> Orchestre le `SchemaValidator` + les 6 dimensions en un seul appel.  
> Retourne un rapport structuré complet.

In [ ]:
def run_pipeline_dq(df: pd.DataFrame,
                     nom_extraction: str,
                     colonnes_critiques: list,
                     cles_primaires: list,
                     schema_validator: SchemaValidator,
                     df_reference: pd.DataFrame = None,
                     cle_jointure: str = None,
                     col_date_ref: str = None) -> dict:
    """
    Pipeline Data Quality complet SGCI.
    Enchaîne SchemaValidator + les 6 dimensions DQ.

    Args:
        df                 : DataFrame extrait de Teradata
        nom_extraction     : Nom descriptif (ex: 'transactions_J1_ABJ')
        colonnes_critiques : Colonnes bloquantes si nulles
        cles_primaires     : Colonnes formant la clé unique
        schema_validator   : Instance SchemaValidator préconfigurée
        df_reference       : Référentiel pour contrôle inter-tables (optionnel)
        cle_jointure       : Colonne de jointure avec df_reference (optionnel)
        col_date_ref       : Colonne de date pour contrôle fraîcheur (optionnel)

    Returns:
        dict : Rapport complet avec statut, résultats par dimension, durée
    """
    t0 = time.time()

    rapport = {
        "extraction"  : nom_extraction,
        "timestamp"   : datetime.now().isoformat(),
        "date_donnees": str((datetime.now(TZ_ABIDJAN) - timedelta(days=1)).date()),
        "nb_lignes"   : len(df),
        "nb_colonnes" : df.shape[1],
        "valide"      : True,
        "dimensions"  : {},
        "erreurs"     : [],
        "warnings"    : []
    }

    logger.info("═" * 55)
    logger.info(f"PIPELINE DQ — {nom_extraction.upper()}")
    logger.info(f"{len(df):,} lignes · {df.shape[1]} colonnes")
    logger.info("═" * 55)

    df_w = df.copy()

    # ── SchemaValidator ───────────────────────────────────────────────────────
    logger.info("── SchemaValidator")
    sv_result = schema_validator.validate()
    sv_statut = "ECHEC" if sv_result["status"] == "FAIL" else (
                "AVERTISSEMENT" if sv_result["warnings"] else "OK")
    rapport["dimensions"]["schema"] = sv_statut
    if sv_statut == "ECHEC":
        rapport["valide"] = False
        rapport["erreurs"] += sv_result["errors"]
    rapport["warnings"] += sv_result["warnings"]

    # ── 6 Dimensions ─────────────────────────────────────────────────────────
    etapes = [
        ("completude", lambda: verifier_completude(df_w, colonnes_critiques)),
        ("unicite",    lambda: verifier_unicite(df_w, cles_primaires)),
        ("validite",   lambda: verifier_validite(df_w)),
        ("exactitude", lambda: verifier_exactitude(df_w)),
        ("coherence",  lambda: verifier_coherence(df_w, df_reference, cle_jointure)),
        ("fraicheur",  lambda: verifier_fraicheur(df_w, col_date_ref)
                               if col_date_ref else {"dimension": "Fraicheur", "statut": "SKIP"}),
    ]

    for nom_dim, fn in etapes:
        logger.info(f"── {nom_dim.capitalize()}")
        res    = fn()
        statut = res.get("statut", "SKIP")
        rapport["dimensions"][nom_dim] = statut
        if statut == "ECHEC":
            rapport["valide"] = False
            rapport["erreurs"].append(f"{nom_dim.upper()} : ECHEC")
        elif statut == "AVERTISSEMENT":
            rapport["warnings"].append(f"{nom_dim.upper()} : AVERTISSEMENT")

    rapport["duration_s"] = round(time.time() - t0, 2)

    # ── Log résumé ────────────────────────────────────────────────────────────
    icones  = {"OK": "✅", "ECHEC": "❌", "AVERTISSEMENT": "⚠️", "SKIP": "⏭️"}
    verdict = "✅ VALIDE" if rapport["valide"] else "❌ NON VALIDE"
    logger.info("─" * 55)
    for dim, st in rapport["dimensions"].items():
        logger.info(f"  {dim.capitalize():<22} {icones.get(st,'?')} {st}")
    logger.info("─" * 55)
    logger.info(f"  RÉSULTAT : {verdict} | Durée : {rapport['duration_s']}s")
    logger.info("═" * 55)

    return rapport


print("✅ run_pipeline_dq() défini")


In [ ]:
# ═══════════════════════════════════════════════════════════
#  EXÉCUTION DU PIPELINE
# ═══════════════════════════════════════════════════════════

COLONNES_REQUISES  = ["TRN_REF","ACCOUNT_ID","AMOUNT","CURRENCY",
                       "TRANSACTION_TYPE","TRANSACTION_SIDE","VALUE_DATE","BOOKING_DATE"]
COLONNES_CRITIQUES = ["TRN_REF","ACCOUNT_ID","AMOUNT","VALUE_DATE","CURRENCY"]
CLES_PRIMAIRES     = ["TRN_REF"]

# Configurer le SchemaValidator
sv = (SchemaValidator("transactions_J1", logger)
      .check_required_columns(df_raw, COLONNES_REQUISES)
      .check_no_null(df_raw, ["TRN_REF","ACCOUNT_ID","AMOUNT"])
      .check_positive(df_raw, "AMOUNT")
      .check_positive(df_raw, "BALANCE")
      .check_range(df_raw, "AMOUNT", 0, 500_000_000)
      .check_unique_by(df_raw, CLES_PRIMAIRES)
      .check_no_full_duplicates(df_raw)
)

# Lancer le pipeline complet
rapport = run_pipeline_dq(
    df                 = df_raw,
    nom_extraction     = "transactions_J1_ABJ",
    colonnes_critiques = COLONNES_CRITIQUES,
    cles_primaires     = CLES_PRIMAIRES,
    schema_validator   = sv,
    df_reference       = df_comptes,
    cle_jointure       = "ACCOUNT_ID",
    col_date_ref       = "BOOKING_DATE"
)


## 5. Journalisation <a id='5'></a>

In [ ]:
# ═══════════════════════════════════════════════════════════
#  FONCTION LOG — JOURNAL CSV STRUCTURÉ
#  Enregistre chaque exécution de manière cumulative.
#  Format : séparateur point-virgule, encodage UTF-8.
# ═══════════════════════════════════════════════════════════
def log(log_dir: str, log_file_name: str, Execution_Date: str, User: str,
        Project: str, File_Name: str, Frequency: str, Data_Begin: str,
        Data_End: str, File_Size: float, Row_Number: int, Column_Number: int,
        Duration: float, Status: str, Error_Reason: str) -> None:
    """
    Ajoute une entrée dans le journal CSV cumulatif.

    Args:
        log_dir       : Répertoire du fichier journal
        log_file_name : Nom du fichier CSV (ex: 'journal_dq.csv')
        Execution_Date: Horodatage ISO 8601
        User          : Identifiant utilisateur système
        Project       : Nom du projet / pipeline
        File_Name     : Nom de l'extraction ou du fichier traité
        Frequency     : 'Quotidien' | 'Hebdomadaire' | 'Mensuel' | 'Trimestriel' | 'Semestriel'
        Data_Begin    : Date de début des données (YYYY-MM-DD)
        Data_End      : Date de fin des données (YYYY-MM-DD)
        File_Size     : Taille en Ko
        Row_Number    : Nombre de lignes
        Column_Number : Nombre de colonnes
        Duration      : Durée d'exécution en secondes
        Status        : 'Good' | 'Bad'
        Error_Reason  : Description des erreurs (vide si succès)
    """
    record = [{
        "Execution_Date" : Execution_Date,
        "User"           : User,
        "Project"        : Project,
        "File_Name"      : File_Name,
        "Frequency"      : Frequency,
        "Data_Begin"     : Data_Begin,
        "Data_End"       : Data_End,
        "File_Size(Ko)"  : File_Size,
        "Row_Number"     : Row_Number,
        "Column_Number"  : Column_Number,
        "Duration(s)"    : Duration,
        "Status"         : Status,
        "Error_Reason"   : Error_Reason
    }]

    df_new    = pd.DataFrame(record)
    full_path = os.path.join(log_dir, log_file_name)

    # Historique cumulatif : concaténer si le fichier existe déjà
    if os.path.isfile(full_path):
        df_old = pd.read_csv(full_path, sep=";", encoding="utf-8")
        df_new = pd.concat([df_old, df_new], ignore_index=True)

    df_new.to_csv(full_path, sep=";", index=False, encoding="utf-8")
    logger.debug(f"Journal mis à jour → {full_path} ({len(df_new)} entrée(s))")


print("✅ Fonction log() définie")


In [ ]:
# ═══════════════════════════════════════════════════════════
#  JOURNALISATION DU RÉSULTAT DU PIPELINE
# ═══════════════════════════════════════════════════════════
hier        = (datetime.now(TZ_ABIDJAN) - timedelta(days=1)).date()
statut_log  = "Good" if rapport["valide"] else "Bad"
erreurs_log = " | ".join(rapport["erreurs"]) if rapport["erreurs"] else ""
taille_ko   = round(df_raw.memory_usage(deep=True).sum() / 1024, 2)

log(
    log_dir        = LOG_DIR,
    log_file_name  = LOG_FILE_NAME,
    Execution_Date = rapport["timestamp"],
    User           = USER,
    Project        = "SGCI_DQ",
    File_Name      = rapport["extraction"],
    Frequency      = "Quotidien",
    Data_Begin     = str(hier),
    Data_End       = str(hier),
    File_Size      = taille_ko,
    Row_Number     = rapport["nb_lignes"],
    Column_Number  = rapport["nb_colonnes"],
    Duration       = rapport["duration_s"],
    Status         = statut_log,
    Error_Reason   = erreurs_log
)

# Lecture et affichage du journal
df_journal = pd.read_csv(os.path.join(LOG_DIR, LOG_FILE_NAME), sep=";")
logger.info(f"Journal mis à jour : {len(df_journal)} entrée(s) au total")
print(f"\n📋 Journal — {len(df_journal)} entrée(s)")
df_journal.tail(5)


In [ ]:
# ═══════════════════════════════════════════════════════════
#  EXPORT RAPPORT EXCEL
#  4 onglets : Résumé · Dimensions · Erreurs · Journal
# ═══════════════════════════════════════════════════════════
def exporter_rapport_excel(rapport: dict, df_journal: pd.DataFrame,
                            log_dir: str = LOG_DIR) -> str:
    """
    Exporte le rapport DQ dans un fichier Excel structuré.

    Onglets :
        Résumé      → Vue synthétique du pipeline
        Dimensions  → Statut détaillé par dimension
        Erreurs     → Liste des erreurs et warnings
        Journal     → Historique complet des exécutions
    """
    ts    = datetime.now().strftime("%Y%m%d_%H%M")
    fname = os.path.join(log_dir, f"rapport_dq_{rapport['extraction']}_{ts}.xlsx")

    icones = {"OK": "✅", "ECHEC": "❌", "AVERTISSEMENT": "⚠️", "SKIP": "⏭️"}

    with pd.ExcelWriter(fname, engine="openpyxl") as writer:

        # ── Onglet 1 : Résumé ──────────────────────────────────────────────────
        pd.DataFrame([{
            "Extraction"     : rapport["extraction"],
            "Horodatage"     : rapport["timestamp"],
            "Date données"   : rapport["date_donnees"],
            "Nb lignes"      : rapport["nb_lignes"],
            "Nb colonnes"    : rapport["nb_colonnes"],
            "Durée (s)"      : rapport["duration_s"],
            "Résultat"       : "✅ VALIDE" if rapport["valide"] else "❌ NON VALIDE",
            "Nb erreurs"     : len(rapport["erreurs"]),
            "Nb warnings"    : len(rapport["warnings"]),
        }]).to_excel(writer, sheet_name="Résumé", index=False)

        # ── Onglet 2 : Dimensions ──────────────────────────────────────────────
        pd.DataFrame([
            {
                "Dimension" : k.capitalize(),
                "Statut"    : v,
                "Icône"     : icones.get(v, "?"),
                "Bloquant"  : "OUI" if v == "ECHEC" else "NON"
            }
            for k, v in rapport["dimensions"].items()
        ]).to_excel(writer, sheet_name="Dimensions", index=False)

        # ── Onglet 3 : Erreurs & Warnings ─────────────────────────────────────
        lignes_ew = (
            [{"Type": "ERREUR",   "Message": e} for e in rapport["erreurs"]] +
            [{"Type": "WARNING",  "Message": w} for w in rapport["warnings"]]
        )
        if lignes_ew:
            pd.DataFrame(lignes_ew).to_excel(writer, sheet_name="Erreurs & Warnings", index=False)
        else:
            pd.DataFrame([{"Type": "—", "Message": "Aucune anomalie détectée"}]).to_excel(
                writer, sheet_name="Erreurs & Warnings", index=False)

        # ── Onglet 4 : Journal complet ─────────────────────────────────────────
        df_journal.to_excel(writer, sheet_name="Journal", index=False)

    logger.info(f"Rapport Excel exporté → {fname}")
    return fname


fname = exporter_rapport_excel(rapport, df_journal)
print(f"\n✅ Rapport exporté : {fname}")
print(f"   Onglets : Résumé · Dimensions · Erreurs & Warnings · Journal")


In [ ]:
# ═══════════════════════════════════════════════════════════
#  VISUALISATION RÉCAPITULATIVE
# ═══════════════════════════════════════════════════════════
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    dims    = list(rapport["dimensions"].items())
    labels  = [d[0].capitalize() for d in dims]
    palette = {"OK":"#1B7A3E","ECHEC":"#E3000F","AVERTISSEMENT":"#E67E22",
               "SKIP":"#95A5A6","FAIL":"#E3000F"}
    icones  = {"OK":"✅","ECHEC":"❌","AVERTISSEMENT":"⚠️","SKIP":"⏭️"}
    couleurs= [palette.get(d[1],"#95A5A6") for d in dims]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Rapport DQ — {rapport['extraction']} — {date.today():%d/%m/%Y}",
        fontsize=13, fontweight="bold"
    )

    # ── Gauche : statut par dimension ─────────────────────────────────────────
    bars = axes[0].barh(labels, [1]*len(labels), color=couleurs,
                         edgecolor="white", height=0.55)
    for bar, (_, st) in zip(bars, dims):
        axes[0].text(0.5, bar.get_y()+bar.get_height()/2,
                     icones.get(st,"?"), va="center", ha="center", fontsize=14)
    axes[0].set_xlim(0, 1); axes[0].set_xticks([])
    axes[0].set_title("Statut par dimension", fontweight="bold")
    axes[0].spines[["top","right","bottom","left"]].set_visible(False)
    leg = [mpatches.Patch(color=c, label=l) for l,c in
           [("✅ OK","#1B7A3E"),("⚠️ Warning","#E67E22"),("❌ Erreur","#E3000F")]]
    axes[0].legend(handles=leg, loc="lower right", fontsize=9, framealpha=0.9)

    # ── Droite : résumé chiffré ────────────────────────────────────────────────
    axes[1].axis("off")
    resume = [
        ("Extraction",      rapport["extraction"]),
        ("Date données",    rapport["date_donnees"]),
        ("Nb lignes",       f"{rapport['nb_lignes']:,}"),
        ("Durée DQ",        f"{rapport['duration_s']}s"),
        ("Nb erreurs",      str(len(rapport["erreurs"]))),
        ("Nb warnings",     str(len(rapport["warnings"]))),
        ("Résultat global", "✅ VALIDE" if rapport["valide"] else "❌ NON VALIDE"),
    ]
    y = 0.95
    for label, valeur in resume:
        axes[1].text(0.05, y, f"{label} :", fontsize=10, fontweight="bold",
                     transform=axes[1].transAxes, va="top")
        axes[1].text(0.45, y, valeur, fontsize=10,
                     transform=axes[1].transAxes, va="top",
                     color="#E3000F" if "❌" in valeur else "#1B7A3E" if "✅" in valeur else "#1A1A1A")
        y -= 0.13

    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, f"rapport_dq_visual_{datetime.now():%Y%m%d_%H%M}.png"),
                dpi=120, bbox_inches="tight")
    plt.show()
    print("✅ Visualisation sauvegardée dans JOURNAL/")

except ImportError:
    print("matplotlib non disponible — pip install matplotlib")


---
## 📌 Récapitulatif

### Structure du notebook

```
0. Configuration    → Imports, constantes, référentiels métier SGCI
1. Connexion        → get_connection() Teradata + requêtes SQL prêtes
2. SchemaValidator  → 8 contrôles structurels chaînables
3. 6 Dimensions DQ  → Complétude · Unicité · Validité · Exactitude · Cohérence · Fraîcheur
4. Pipeline         → run_pipeline_dq() — un seul appel orchestre tout
5. Journalisation   → log() CSV + export Excel 4 onglets + visualisation
```

### Passer en production

```python
# Étape 1 : définir les credentials (une seule fois)
os.environ["TD_HOST"]     = "mon_serveur_td"
os.environ["TD_USER"]     = "mon_user"
os.environ["TD_PASSWORD"] = "mon_mdp"

# Étape 2 : remplacer la simulation par les vraies extractions
df_raw, df_comptes = simuler_extraction()          # ← DÉMO
# Par :
df_raw     = extraire_depuis_teradata(QUERY_TRANSACTIONS, "transactions_J1")
df_comptes = extraire_depuis_teradata(QUERY_COMPTES, "referentiel_comptes")
```

### Fichiers générés dans `./JOURNAL/`

| Fichier | Contenu |
|---|---|
| `sgci_dq_framework.log` | Log texte complet (tous niveaux) |
| `journal_dq.csv` | Journal CSV cumulatif de toutes les exécutions |
| `rapport_dq_*.xlsx` | Rapport Excel horodaté (4 onglets) |
| `rapport_dq_visual_*.png` | Graphique récapitulatif |

---
*SGCI Data Engineering · Framework DQ v3.0 · Juin 2026*
